In [1]:
import numpy as np
import pandas as pd
import glob

import xarray as xr

from pathlib import Path

In [2]:
from __future__ import annotations

from pathlib import Path
from utils.event_iden_funcs import build_events_df

- ERA5 temperature file for a given level (for myself)

In [5]:
# import module path
import sys
sys.path.insert(1, '/net/cfc/s2s/rachwu/scripts/tools/read_hc/')
sys.path.insert(2, '/net/cfc/s2s/rachwu/scripts/tools/read_data/')
sys.path.insert(3, '/net/cfc/s2s/rachwu/scripts/WP2/modules')
sys.path.insert(4, '/net/cfc/s2s/rachwu/scripts/tools/wind_events/')

In [6]:
import mod_read_erai as mera

In [7]:
def return_u300(start_date, end_date, ilat, ilon, ilev, varname):
    time_gph, lon, lat, lev, u_era5 = mera.return_var_era5(varname, start_date, end_date, ilat, ilon, ilev)
    u_era5 = u_era5.groupby(u_era5.time.dt.floor('1D')).mean().rename({'floor':'time'})
    time_era5 = u_era5.time
    
    return time_era5, lat, lon, u_era5

In [17]:
def build_multiyear_T(
    start_year=1979,
    end_year=2023,
    ilat=slice(0,-90), 
    ilon=slice(0,360),
    ilev=1000,
    varname="geopot",
    out_dir="./processed",
    overwrite=False,
):
    """
    Read ERA5 wind data year by year, compute daily means, and merge into one file.
    Saves as: era5_u_<level>hPa_<start>_<end>.nc
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    outfile = out_dir / f"era5_{varname}_{int(ilev/100)}hPa_{start_year}_{end_year}.nc"

    if outfile.exists() and not overwrite:
        print(f" Found existing file: {outfile}")
        return xr.open_dataarray(outfile)

    all_years = []

    for yr in range(start_year, end_year + 1):
        print(f"Processing ERA5 {varname} for {yr}...")
        try:
            time_era5, lat, lon, u_yr = return_u300(f"{yr}-01-01", f"{yr}-12-31", ilat=ilat, ilon=ilon, ilev=ilev, varname=varname)
            all_years.append(u_yr)
        except Exception as e:
            print(f"Skipping {yr}: {e}")

    # Combine along time dimension
    u_all = xr.concat(all_years, dim="time").sortby("time")

    # Preserve the level as a coordinate, even if it’s a single value
    u_all = u_all.expand_dims("plev") if "plev" not in u_all.dims else u_all
    u_all = u_all.assign_coords(plev=[ilev])


    # Ensure plev is second in dimension order
    new_order = ["time", "plev", "lat", "lon"]
    u_all = u_all.transpose(*[d for d in new_order if d in u_all.dims])
    
    # Save
    u_all.to_netcdf(outfile)
    print(f" Saved merged file: {outfile}")

    return u_all


In [18]:
T_all = build_multiyear_T(
    start_year=1979,
    end_year=2023,
    ilev=1000,
    out_dir="./processed",
    overwrite=False,
)

Processing ERA5 geopot for 1979...
Processing ERA5 geopot for 1980...
Processing ERA5 geopot for 1981...
Processing ERA5 geopot for 1982...
Processing ERA5 geopot for 1983...
Processing ERA5 geopot for 1984...
Processing ERA5 geopot for 1985...
Processing ERA5 geopot for 1986...
Processing ERA5 geopot for 1987...
Processing ERA5 geopot for 1988...
Processing ERA5 geopot for 1989...
Processing ERA5 geopot for 1990...
Processing ERA5 geopot for 1991...
Processing ERA5 geopot for 1992...
Processing ERA5 geopot for 1993...
Processing ERA5 geopot for 1994...
Processing ERA5 geopot for 1995...
Processing ERA5 geopot for 1996...
Processing ERA5 geopot for 1997...
Processing ERA5 geopot for 1998...
Processing ERA5 geopot for 1999...
Processing ERA5 geopot for 2000...
Processing ERA5 geopot for 2001...
Processing ERA5 geopot for 2002...
Processing ERA5 geopot for 2003...
Processing ERA5 geopot for 2004...
Processing ERA5 geopot for 2005...
Processing ERA5 geopot for 2006...
Processing ERA5 geop

- u

In [40]:
#!/usr/bin/env python3
"""
Build multi-year ERA5 u-wind dataset (daily or monthly means, any pressure levels).
"""

import xarray as xr
from pathlib import Path
import numpy as np

def build_multiyear_u(
    start_year=1979,
    end_year=2023,
    ilev=None,                   # e.g., 10000 for 100 hPa, or None for all levels
    ilat=slice(-90, -20),
    ilon=slice(0, 360),
    varname="uwind",
    out_dir="./processed",
    overwrite=False,
    time_avg="daily",            # 'daily', 'monthly', or None (keep 6-hourly)
):
    """
    Read ERA5 u-wind data year by year, average to daily or monthly means, and merge.

    Parameters
    ----------
    start_year, end_year : int
        Time range to process.
    ilev : float or None
        Pressure level in Pa (e.g. 10000 = 100 hPa). If None, take all levels.
    ilat, ilon : slice
        Spatial selection.
    varname : str
        Variable name in dataset (e.g. "u", "uwind").
    time_avg : str or None
        Temporal averaging:
          - 'daily'   → 24h mean
          - 'monthly' → calendar month mean
          - None      → keep original 6-hourly resolution
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # Name output file
    lv_str = "all_lv" if ilev is None else f"{int(ilev/100)}hPa"
    outfile = out_dir / f"era5_{varname}_{lv_str}_{time_avg}_{start_year}_{end_year}.nc"

    if outfile.exists() and not overwrite:
        print(f" Found existing file: {outfile}")
        return xr.open_dataarray(outfile)

    all_years = []

    for yr in range(start_year, end_year + 1):
        print(f"Processing ERA5 {varname} for {yr}...")
        try:
            # --- Load one year (your function should support ilev=None)
            time_era5, lat, lon, u_yr = return_u300(
                f"{yr}-01-01", f"{yr}-12-31",
                ilat=ilat, ilon=ilon, ilev=ilev, varname=varname
            )

            # --- Temporal averaging
            if time_avg == "daily":
                u_yr = u_yr.resample(time="1D").mean()
            elif time_avg == "monthly":
                u_yr = u_yr.resample(time="1MS").mean()

            all_years.append(u_yr)

        except Exception as e:
            print(f"Skipping {yr}: {e}")
            continue

    # --- Combine along time dimension
    u_all = xr.concat(all_years, dim="time").sortby("time")

    # --- Keep plev coordinate if available
    if "plev" not in u_all.dims and ilev is not None:
        u_all = u_all.expand_dims("plev").assign_coords(plev=[ilev])

    # --- Ensure standard dimension order
    new_order = [d for d in ["time", "plev", "lat", "lon"] if d in u_all.dims]
    u_all = u_all.transpose(*new_order)

    # --- Metadata
    u_all.name = varname
    u_all.attrs.update(dict(
        description=f"ERA5 {varname} ({time_avg} mean)",
        years=f"{start_year}-{end_year}",
        level="all" if ilev is None else f"{int(ilev/100)} hPa",
        units="m s⁻¹",
    ))

    # --- Save
    u_all.to_netcdf(outfile)
    print(f"✅ Saved merged file: {outfile}")

    return u_all


In [41]:
ilat = slice(-55,-65)

In [42]:
# All levels, monthly mean
u_mon = build_multiyear_u(
    start_year=1979,
    end_year=2023,
    ilev=None,                # all levels
    ilat=ilat,
    varname="uwind",
    time_avg="monthly",
    out_dir="./processed/u_mon"
)


Processing ERA5 uwind for 1979...
Processing ERA5 uwind for 1980...
Processing ERA5 uwind for 1981...
Processing ERA5 uwind for 1982...
Processing ERA5 uwind for 1983...
Processing ERA5 uwind for 1984...
Processing ERA5 uwind for 1985...
Processing ERA5 uwind for 1986...
Processing ERA5 uwind for 1987...
Processing ERA5 uwind for 1988...
Processing ERA5 uwind for 1989...
Processing ERA5 uwind for 1990...
Processing ERA5 uwind for 1991...
Processing ERA5 uwind for 1992...
Processing ERA5 uwind for 1993...
Processing ERA5 uwind for 1994...
Processing ERA5 uwind for 1995...
Processing ERA5 uwind for 1996...
Processing ERA5 uwind for 1997...
Processing ERA5 uwind for 1998...
Processing ERA5 uwind for 1999...
Processing ERA5 uwind for 2000...
Processing ERA5 uwind for 2001...
Processing ERA5 uwind for 2002...
Processing ERA5 uwind for 2003...
Processing ERA5 uwind for 2004...
Processing ERA5 uwind for 2005...
Processing ERA5 uwind for 2006...
Processing ERA5 uwind for 2007...
Processing ERA

In [29]:
yr=1979
ilon=slice(0,360)
ilev=None
varname='uwind'

In [30]:
time_era5, lat, lon, u_yr = return_u300(
                f"{yr}-01-01", f"{yr}-12-31",
                ilat=ilat, ilon=ilon, ilev=ilev, varname=varname
            )

In [39]:
u_yr.resample(time="1MS").mean()

<xarray.DataArray 'var131' (time: 12, plev: 37, lat: 5, lon: 180)> Size: 2MB
array([[[[-2.54910965e+01, -2.55704823e+01, -2.57764416e+01, ...,
          -2.56806450e+01, -2.57946815e+01, -2.56477242e+01],
         [-2.47230415e+01, -2.47778873e+01, -2.49059734e+01, ...,
          -2.50242958e+01, -2.50360470e+01, -2.48568001e+01],
         [-2.47463531e+01, -2.48557606e+01, -2.49606037e+01, ...,
          -2.48916416e+01, -2.48285103e+01, -2.47395172e+01],
         [-2.50398617e+01, -2.51631927e+01, -2.52108231e+01, ...,
          -2.49271793e+01, -2.48787613e+01, -2.49162483e+01],
         [-2.47493496e+01, -2.48048878e+01, -2.47967300e+01, ...,
          -2.46163807e+01, -2.46036530e+01, -2.46578369e+01]],

        [[-2.18141346e+01, -2.19112225e+01, -2.18869362e+01, ...,
          -2.17984142e+01, -2.16320210e+01, -2.16395187e+01],
         [-2.04769688e+01, -2.05933056e+01, -2.05802002e+01, ...,
          -2.04190044e+01, -2.02503109e+01, -2.02844906e+01],
         [-2.02365761e+01, -2.02528000e+01, -2.01913395e+01, ...,
          -2.00873508e+01, -2.00531082e+01, -2.01271381e+01],
         [-2.06662636e+01, -2.06260357e+01, -2.05933990e+01, ...,
          -2.04482059e+01, -2.05407600e+01, -2.06366520e+01],
         [-2.08474007e+01, -2.08514957e+01, -2.08629932e+01, ...,
...
         [ 4.73046494e+00,  4.78768826e+00,  4.77401638e+00, ...,
           4.37252331e+00,  4.48496962e+00,  4.60851240e+00],
         [ 3.12221622e+00,  3.24718475e+00,  3.32706594e+00, ...,
           2.54477644e+00,  2.71447778e+00,  2.92639184e+00],
         [ 1.06045640e+00,  1.23816717e+00,  1.38594317e+00, ...,
           7.08854437e-01,  8.00651431e-01,  9.10254836e-01],
         [-1.58914649e+00, -1.46648550e+00, -1.41821659e+00, ...,
          -1.40687585e+00, -1.54510665e+00, -1.63290286e+00]],

        [[ 4.61670494e+00,  4.49014187e+00,  4.38370895e+00, ...,
           4.82832003e+00,  4.81312037e+00,  4.72632027e+00],
         [ 3.91529703e+00,  3.94780707e+00,  3.91757298e+00, ...,
           3.71411729e+00,  3.79246664e+00,  3.84802842e+00],
         [ 2.53278375e+00,  2.64179659e+00,  2.71414876e+00, ...,
           2.02216768e+00,  2.17096353e+00,  2.35884142e+00],
         [ 6.87336802e-01,  8.46020281e-01,  9.87255633e-01, ...,
           3.01062971e-01,  4.00884986e-01,  5.33606946e-01],
         [-1.58458436e+00, -1.45087409e+00, -1.35817552e+00, ...,
          -1.39148879e+00, -1.53620124e+00, -1.62604499e+00]]]],
      dtype=float32)
Coordinates:
  * lon      (lon) float64 1kB 0.0 2.0 4.0 6.0 8.0 ... 352.0 354.0 356.0 358.0
  * lat      (lat) float64 40B -56.0 -58.0 -60.0 -62.0 -64.0
  * plev     (plev) float64 296B 100.0 200.0 300.0 ... 9.5e+04 9.75e+04 1e+05
  * time     (time) datetime64[ns] 96B 1979-01-01 1979-02-01 ... 1979-12-01
Attributes:
    table:    128